**Objective**

Before answering business questions pertaining to this dataset, clean the `customers` and `orders` tables to address data quality and fix potential issues. In addition, investigate the orders table and understand the relationship between `order_status` and missing datetime values. Lastly, investigate the chronological order of the timestamps in the `orders` table.


In [1]:
# Import libraries and load datasets

import pandas as pd

customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")

**Customers Table Data Cleaning**


In [2]:
# Convert zip code prefix to string

customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].astype(str)

In [3]:
# Verify change

customers.dtypes

,0
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,object
customer_city,object
customer_state,object


**Analyst Commentary**

Conversion verified: `customer_zip_code_prefix` is now a `string`.

**Orders Table Data Cleaning**

In [4]:
# Convert datetime columns from string to datetime

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"], errors="coerce")
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"], errors="coerce")
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"], errors="coerce")
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"], errors="coerce")

In [5]:
# Verify change

orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


**Analyst Commentary**

Conversion verified: `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date` are all in `datetime`.

**Orders Table Investigation**

The goal of this investigation is to evaluate the relationship between `order_status` and missing datetime values.

In [6]:
# Which statuses have missing approval timestamps?

orders[orders["order_approved_at"].isnull()].groupby("order_status").size()

,0
order_status,
canceled,141
created,5
delivered,14


**Analyst Commentary**

1. 141 canceled orders have missing approval timestamps makes sense. These orders were canceled before it was approved for processing.

2. 5 created orders have missing approval timestamps. These orders were created, but haven't been approved yet. This makes sense in the fulfillment context.

3. 14 delivered orders have missing approval timestamps. If these orders were delivered, then it passed through the entirety of the fulfillment process. However, these records have missing approval timestamps. Need to investigate why.

In [7]:
# What is going on with those 14 delivered orders that have missing approval timestamps?

orders[(orders["order_status"] == "delivered") & (orders["order_approved_at"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


**Analyst Commentary**

Fourteen delivered orders are missing `order_approved_at`, but all other fulfillment dates are complete and sequentially consistent. `order_purchase_timestamp` ranged between  mid-January to mid-February. `order_delivered_carrier_date` occurred between late January and late February. All of the orders were delivered in early March. `order_estimated_delivery_date` was around mid-March for all 14 orders. Based on the data provided, this suggests an unknown logging gap at the approval step instead of a broader data integrity issue.

In [8]:
# Isolating the logging gap by month

orders[(orders["order_status"] == "delivered") & (orders["order_approved_at"].isnull())]["order_purchase_timestamp"].dt.to_period("M").value_counts()

,count
order_purchase_timestamp,
2017-02,12
2017-01,2


In [9]:
# Isolating the logging gap by day-of-month

orders[(orders["order_status"] == "delivered") & (orders["order_approved_at"].isnull())]["order_purchase_timestamp"].dt.day.value_counts().sort_index()

,count
order_purchase_timestamp,
17,3
18,8
19,3


**Analyst Commentary**

Fourteen delivered orders (from Jan-Feb 2018) are missing `order_approved_at`. All other fulfillment dates are complete and sequentially valid. The clustering within two specific months suggests a systemic/operational cause (i.e. logging gap or batch process issue during that period) rather than random data entry error.

*Cleaning Recommendation:* Keep records for analysis.

In [17]:
# Which statuses have missing carrier timestamps?


orders[orders["order_delivered_carrier_date"].isnull()].groupby("order_status").size()

,0
order_status,
approved,2
canceled,550
created,5
delivered,2
invoiced,314
processing,301
unavailable,609


**Analyst Commentary**

Most of these make sense structurally. If an order has not progressed far enough in the fulfillment process, it wouldn't have a carrier handoff date yet.

1. created (5), approved (2), invoiced (314), processing (301) are all early-stage statuses. They haven't reached the carrier yet, so a missing carrier date is expected and not an error.

2. canceled (550) is also plausible. If cancelled before shipment, there's be no carrier timestamp to log.

3. unavailable (609) make sense. If the product isn't available, the order will likely never ship.

This needs an investigation:
delivered orders (2) have missing carrier timestamps. If it's delivered, it must have passed through the carrier step, but there's no timestamp.




In [18]:
# What is going on with those 2 delivered orders that have missing carrier timestamps?

orders[(orders["order_status"] == "delivered") & (orders["order_delivered_carrier_date"].isnull())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


**Analyst Commentary**

Row 1 (`order_id` = 2aa911...):

Purchased on September 29th, approved same day, carrier date missing, but delivered on November 20th. That's roughly two months later and 6 days after the estimated delivery date (November 14th).

This suggests that the order did arrive, but the carrier timestamp wasn't recorded. In addition, this record flags a performance issue (the long gap between purchase and delivery and the late arrival versus estimate), which will be discussed in a future analysis.

Similar to the `order_approved_at` finding, the missing carrier date looks like an isolated data gap.


Row 2 (`order_id` = 2d858...):

Purchased and approved on May 25th. `order_delivered_carrier_date` and `order_delivered_customer_date` are missing, but status still says `delivered`. Need to investigate why.

In [19]:
# Is there any other evidence that this order was fulfilled?

orders[orders["order_id"] == "2d858f451373b04fb5c984a1cc2defaf"]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


**Analyst Commentary**

The status is "delivered", but there's no timestamp evidence. Will check if this order_id appears in other related tables (`order_items` or `order_reviews`) to see if there's evidence of fulfillment.

In [20]:
# Importing tables

order_items = pd.read_csv("olist_order_items_dataset.csv")
order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")

# Verifying whether the order_id is present in these datasets
target_id = "2d858f451373b04fb5c984a1cc2defaf"

in_items = target_id in order_items["order_id"].values
in_reviews = target_id in order_reviews["order_id"].values

print(f"Appears in order_items: {in_items}")
print(f"Appears in order_reviews: {in_reviews}")

Appears in order_items: True
Appears in order_reviews: True


In [21]:
# Getting the actual row in order_items table

order_items[order_items["order_id"] == target_id]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
19838,2d858f451373b04fb5c984a1cc2defaf,1,30b5b5635a79548a48d04162d971848f,f9bbdd976532d50b7816d285a22bd01e,2017-06-04 23:30:16,179.0,15.0


In [22]:
# Getting the actual row in order_reviews table

order_reviews[order_reviews["order_id"] == target_id]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
51996,4e755f114e50d33b9ac6a56e0d7d3ea9,2d858f451373b04fb5c984a1cc2defaf,5,NaN,NaN,2017-06-25 00:00:00,2017-06-27 01:49:04


**Analyst Commentary**

Based on the data from the `order_items` and `order_reviews` tables, the order flagged earlier for missing `order_delivered_carrier_date` and `order_delivered_customer_date` has a valid product, seller, price, and received a 5-star review shortly after the estimated delivery window. This suggests that the order was fulfilled, and that the missing fields represent a data logging gap in the `orders` table, not a invalid order.

*Cleaning Recommendation:* Keep both records.

In [23]:
# Which statuses have missing delivery timestamps?

orders[orders["order_delivered_customer_date"].isnull()].groupby("order_status").size()

,0
order_status,
approved,2
canceled,619
created,5
delivered,8
invoiced,314
processing,301
shipped,1107
unavailable,609


**Analyst Commentary**

Orders that have not reached the customer yet do not have a delivery date.

1. created (5), approved (2), invoiced (314), and processing (301) are early-stage fulfillment processes. Haven't been delivered.

2. shipped (1107) make sense. These orders are enroute to their final destination.

3. canceled (619) and unavailable (609) means that they were never completed, so delivery data isn't expected.


This is concerning: delivered (8) means that these orders have missing delivery timestamps. Need to investigate why. In particular, whether if this stems from a logging gap or something else.


In [24]:
# Identify the rows that have a delivered status but missing a order_delivered_customer_date

delivered_missing_date = orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isnull())]
delivered_missing_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


**Analyst Commentary**

`order_id` = 2d858f451373b04fb5c984a1cc2defaf	was fully investigated earlier and was confirmed as a genuine logging gap.

The other 7 entries are only missing `ordered_delivered_customer_date`. These orders left the warehouse (since there's a `order_delivered_carrier_date` timestamp for all 7 orders), but the delivery timestamp is missing.

In [25]:
# The remaining 7
remaining_7 = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull()) &
    (orders["order_delivered_carrier_date"].notnull())
]
target_ids = remaining_7["order_id"]

In [26]:
# Cross-referencing against order_items

items_check = order_items[order_items["order_id"].isin(target_ids)]
items_check

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
5841,0d3268bad9b086af767785e3f0fc0133,1,ec165cd31c50585786ffda6feff5d0a6,8bdd8e3fd58bafa48af76b2c5fd71974,2018-07-05 21:29:54,188.99,15.63
14472,20edc82cf5400ce95e1afacc25798b31,1,55bfa0307d7a46bed72c492259921231,343e716476e3748b069f980efbaa294e,2018-07-03 16:29:30,45.90,9.07
19642,2d1e2d5bf4dc7227b3bfebb81328c15f,1,a50acd33ba7a8da8e9db65094fa990a4,8581055ce74af1daba164fdbd55a40de,2017-12-04 17:56:40,117.30,17.53
20393,2ebdfc4f15f23b91474edf87475f108e,1,e7d5464b94c9a5963f7c686fc80145ad,58f1a6197ed863543e0136bdedb3fce2,2018-07-05 17:15:12,139.00,19.07
75303,ab7c89dc1bf4a1ead9d6ec1ec8968a84,1,a2a7efc985315e86d4f0f705701b342b,ed4acab38528488b65a9a9c603ff024a,2018-06-18 12:30:35,110.99,9.13
101642,e69f75a717d64fc5ecdfae42b2e8e086,1,e7d5464b94c9a5963f7c686fc80145ad,58f1a6197ed863543e0136bdedb3fce2,2018-07-05 22:15:14,139.00,19.07
108192,f5dd62b788049ad9fc0526e3ad11a097,1,2167c8f6252667c0eb9edd51520706a1,0bb738e4d789e63e2267697c42d35a2d,2018-06-26 07:19:05,329.00,25.24


In [27]:
# Cross-referencing against order_reviews

reviews_check = order_reviews[order_reviews["order_id"].isin(target_ids)]
reviews_check

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
10690,ee2d30652e2f7fc00861074f795f5bf0,0d3268bad9b086af767785e3f0fc0133,5,Excelente!,O produto chegou muito antes do prazo previsto...,2018-07-06 00:00:00,2018-07-07 18:48:09
24335,bb311d9562ecbefc8e4be756d8999892,e69f75a717d64fc5ecdfae42b2e8e086,5,NaN,NaN,2018-07-07 00:00:00,2018-07-10 11:38:13
27655,c0dd6bec0375c376f044af102118526f,f5dd62b788049ad9fc0526e3ad11a097,5,Entrega super rápida.,"Produto novo, muito bom.",2018-06-29 00:00:00,2018-06-29 16:26:37
56411,25e11638a3d01a87e8e62338a39eee28,2ebdfc4f15f23b91474edf87475f108e,5,NaN,NaN,2018-07-11 00:00:00,2018-07-11 19:27:46
86011,f48c6c944a5d52dcca8ac5c4ec417cf2,2d1e2d5bf4dc7227b3bfebb81328c15f,5,NaN,Chegou rápido tudo ok,2017-12-19 00:00:00,2017-12-19 04:15:39
88893,d055795a562efffefe47ef81e5435322,20edc82cf5400ce95e1afacc25798b31,5,Muito bom,Adorei,2018-07-06 00:00:00,2018-07-06 20:30:17
92001,0d4c56af896dd6eb9de8edbaa1902d22,ab7c89dc1bf4a1ead9d6ec1ec8968a84,1,Péssimo,Comprei um produto de uma marca e recebi outro...,2018-06-16 00:00:00,2018-06-16 13:55:00


**Analyst Commentary**

All 7 `order_ids` have a real `product_id`, `seller_id`, and price, which suggests that there's evidence of a legitimate, fulfilled transaction and not an error. In addition, all 7 order_ids have a customer review on file. A review is a strong, indirect evidence the customer actually received the product, although there isn't a logged delivered customer date.

One note: There's a one star review. Did the customer say anything about delivery?

In [28]:
# What does that 1-star review say?

order_reviews[order_reviews["order_id"] == "ab7c89dc1bf4a1ead9d6ec1ec8968a84"][["review_comment_title", "review_comment_message"]]

,review_comment_title,review_comment_message
92001,Péssimo,Comprei um produto de uma marca e recebi outro...


**Analyst Commentary**

*Translation*

*Title: Terrible*

*Comment/Message: I bought a* *product from one brand and received another...*

The review discusses receiving the wrong product, not about the delivery. If it was a genuine "never arrived" complaint, then that would be a fulfillment error and would need to be addressed before analysis.

*Cleaning Recommendation:* Keep the records.

**Analyst Commentary**

Based on the analysis, the majority of missing values are consistent with the lifecycle of an order. Delivered orders with missing timestamps were the only inconsistency, confirmed to be data logging gaps rather than lifecycle inconsistencies.

Next, I will check timestamp inconsistencies.

**Chronological Timestamp Validation**

In [29]:
# Are there any records where the purchase date is > approval date?

(orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum()

np.int64(0)

**Analyst Commentary**

0 records found where `order_approved_at` is earlier than `order_purchase_timestamp`. This sequence is valid across the entire dataset.

In [30]:
# Are there any records where the estimated delivery date is < purchase date?

(orders["order_estimated_delivery_date"] < orders["order_purchase_timestamp"]).sum()


np.int64(0)

**Analyst Commentary**

0 records found where `order_estimated_delivery_date` is earlier than `order_purchase_timestamp`. This sequence is valid across the entire dataset.

In [32]:
# Are there any invalid deliveries where the delivered date is < purchase date?

invalid_delivery = orders[
    orders["order_delivered_customer_date"] <
    orders["order_purchase_timestamp"]
]

invalid_delivery

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


**Analyst Commentary**

0 records found where `order_delivered_customer_date` is earlier than `order_purchase_timestamp`. This sequence is valid across the entire dataset.

In [33]:
# Are there any records where the approval date is > carrier date?

(orders["order_delivered_carrier_date"] < orders["order_approved_at"]).sum()

np.int64(1359)

**Analyst Commentary**

1359 orders where the carrier date comes before the approval date. Based on the fulfillment process, that's impossible. Need to investigate what happened.

In [34]:
# Inspecting the 1359 orders

carrier_before_approved = orders[orders["order_delivered_carrier_date"] < orders["order_approved_at"]].copy()
carrier_before_approved[["order_id", "order_status", "order_delivered_carrier_date", "order_approved_at"]]

,order_id,order_status,order_delivered_carrier_date,order_approved_at
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-11 14:54:00,2018-06-12 23:31:02
64,688052146432ef8253587b930b01a06d,delivered,2018-04-23 19:19:14,2018-04-24 18:25:22
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-24 12:57:00,2018-07-26 23:31:53
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:24:00,2018-07-23 12:31:53
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-24 14:03:00,2018-07-27 23:31:09
...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,delivered,2018-07-05 14:11:00,2018-07-05 16:17:59
99230,78008d03bd8ef7fcf1568728b316553c,delivered,2018-07-03 12:57:00,2018-07-05 16:32:52
99266,76a948cd55bf22799753720d4545dd2d,delivered,2018-01-31 18:11:58,2018-02-04 23:31:46
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,delivered,2018-04-23 17:18:40,2018-04-24 19:26:10


In [35]:
# What's the actual gap size?

carrier_before_approved = orders[orders["order_delivered_carrier_date"] < orders["order_approved_at"]].copy()
carrier_before_approved["gap_hours"] = (
    carrier_before_approved["order_approved_at"] - carrier_before_approved["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

carrier_before_approved["gap_hours"].describe()

,gap_hours
count,1359.000000
mean,24.751987
std,115.307793
min,0.005833
25%,1.415417
50%,17.167778
75%,25.956667
max,4109.256111


**Analyst Commentary**

The gap isn't off by a few seconds.

- Median gap: 17.2 hours
- Mean gap: 24.75 hours

Half of these 1359 orders show the carrier date preceding approval by 17+ hours.

The max hour gap is interesting: roughly 4109 hours (~171 days) has a carrier date recorded before the approval timestamp. Need to investigate this further.

In [36]:
# Breaking into severity buckets

print("Rows with gap > 24 hours:", (carrier_before_approved["gap_hours"] > 24).sum())
print("Rows with gap > 1 hour but <= 24 hours:", ((carrier_before_approved["gap_hours"] > 1) & (carrier_before_approved["gap_hours"] <= 24)).sum())
print("Rows with gap <= 1 hour:", (carrier_before_approved["gap_hours"] <= 1).sum())

Rows with gap > 24 hours: 458
Rows with gap > 1 hour but <= 24 hours: 659
Rows with gap <= 1 hour: 242


**Analyst Commentary**

242 orders have a gap in time that is under one hour. It could be a quirk on logging the timestamp.

The rest show gaps ranging from an hour to nearly half a year. Need to check whether the issue here is similar to previous analyses around logging gaps.

In [37]:
# Does this data cluster by time period?

carrier_before_approved["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

,count
order_purchase_timestamp,
2017-02,1
2017-04,24
2017-05,14
2017-06,4
2017-07,21
2017-08,1
2017-09,13
2017-11,1
2017-12,5


**Analyst Commentary**

Three-digit clusters in specific months (April, June, and July of 2018) are a strong, consistent signal that there may be something operational happening during those windows.

In [38]:
# Do these three months overlap with the earlier order_approved_at missing-value clusters?

missing_approval = orders[orders["order_approved_at"].isnull()]
missing_approval["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

,count
order_purchase_timestamp,
2016-10,6
2017-01,3
2017-02,14
2017-03,2
2017-04,4
2017-05,9
2017-06,4
2017-07,5
2017-08,6


**Analyst Commentary**

`order_approved_at` shows two distinct, non-overlapping data quality issues: (1) missing values clustering in Aug–Sep 2018, and (2) logically inverted timestamps (carrier date preceding approval) clustering in Apr, Jun, and Jul 2018. The lack of month overlap suggests these are separate root causes rather than one continuous logging failure — worth investigating as two separate issues.


In [39]:
# Sequence-issue months (carrier date before approval date)

carrier_before_approved = orders[orders["order_delivered_carrier_date"] < orders["order_approved_at"]]
sequence_months = set(carrier_before_approved["order_purchase_timestamp"].dt.to_period("M"))

# Missing-approval months (all statuses)

missing_approval = orders[orders["order_approved_at"].isnull()]
missing_months = set(missing_approval["order_purchase_timestamp"].dt.to_period("M"))

overlap = missing_months & sequence_months
only_missing = missing_months - sequence_months
only_sequence = sequence_months - missing_months

print("Overlapping month(s):", sorted(overlap))
print("Missing-only month(s):", sorted(only_missing))
print("Sequence-issue-only month(s):", sorted(only_sequence))

Overlapping month(s): [Period('2017-02', 'M'), Period('2017-04', 'M'), Period('2017-05', 'M'), Period('2017-06', 'M'), Period('2017-07', 'M'), Period('2017-08', 'M'), Period('2017-09', 'M'), Period('2017-11', 'M'), Period('2017-12', 'M'), Period('2018-01', 'M'), Period('2018-02', 'M'), Period('2018-03', 'M'), Period('2018-05', 'M'), Period('2018-07', 'M'), Period('2018-08', 'M')]
Missing-only month(s): [Period('2016-10', 'M'), Period('2017-01', 'M'), Period('2017-03', 'M'), Period('2017-10', 'M'), Period('2018-09', 'M'), Period('2018-10', 'M')]
Sequence-issue-only month(s): [Period('2018-04', 'M'), Period('2018-06', 'M')]


**Analyst Commentary**

`order_approved_at` shows repeated chronological inconsistencies across much of the observed timeframe (15 of ~21 months affected by both missing values and/or sequencing errors), rather than two isolated, separate incidents. This points to a systemic pattern in how this field is captured. Recommend prioritizing this field specifically for a deeper pipeline review with the engineering team.

Recommendation: For the purposes of this project, keep rows since there's a delivery date and an order_id that connects to related tables. However, this will limit analysis in the future (such as time-series analysis) if the `order_approved_at` field isn't addressed.

In [40]:
# Are there any records where the carrier date is > delivery date?

(orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()


np.int64(23)

**Analyst Commentary**

23 orders where the delivery date comes before the carrier date. That's also impossible based on the current fulfillment process. Need to investigate what happened.

In [41]:
# Inspecting the 23 orders

delivered_before_carrier = orders[orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]]
delivered_before_carrier[["order_id", "order_delivered_carrier_date", "order_delivered_customer_date"]]

,order_id,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-08-09 18:18:43,2017-08-01 21:13:01


In [42]:
# What is the gap for these orders?

delivered_before_carrier = orders[orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]].copy()
delivered_before_carrier["gap_hours"] = (
    delivered_before_carrier["order_delivered_carrier_date"] - delivered_before_carrier["order_delivered_customer_date"]
).dt.total_seconds() / 3600

delivered_before_carrier[["order_id", "order_status", "order_delivered_carrier_date",
                          "order_delivered_customer_date", "gap_hours"]].sort_values("gap_hours", ascending=False)

,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,gap_hours
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-03-02 17:34:26,2017-02-14 15:15:57,386.308056
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01,189.095000
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-06-09 15:07:29,2017-06-02 11:09:16,171.970278
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10,144.238889
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-26 11:43:06,2016-10-20 18:03:17,137.663611
71227,19feb5627c41ea1b36a8e50a469b3644,delivered,2016-10-26 11:42:05,2016-10-20 19:07:54,136.569722
74967,d5558a097766363b8e76b38c43332e8a,delivered,2017-02-15 08:55:26,2017-02-10 07:58:32,120.948333
41636,b866af202be0692766081310cd4085e1,delivered,2017-02-20 02:32:08,2017-02-15 03:53:46,118.639444
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56,69.417222
78556,ea1dcb4757a844d2642547797bd5feb0,delivered,2017-07-27 19:21:31,2017-07-25 19:43:10,47.639167


**Analyst Commentary**

Gap sizes have a wide range from 23 minutes to over 16 days (~386 hours).

The timestamps are valid, but they're chronologically sitting in the wrong column. This suggests a potential data pipeline or ETL issue. Need to investigate further.

In [43]:
# Check if these cluster by time

delivered_before_carrier["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

,count
order_purchase_timestamp,
2016-10,4
2017-01,1
2017-02,2
2017-03,1
2017-05,1
2017-06,1
2017-07,11
2017-08,2


**Analyst Commentary**

2017-07 is interesting. Eleven out of 23 rows (nearly half) were impacted during the month of July 2017.

This is consistent with the "swapped" columns during a specific batch/import" hypothesis.

In [44]:
# Confirm the "swapped columns" hypothesis directly

test_swap = delivered_before_carrier.copy()
test_swap["carrier_swapped"] = test_swap["order_delivered_customer_date"]
test_swap["delivered_swapped"] = test_swap["order_delivered_carrier_date"]

test_swap["sequence_valid_after_swap"] = (
    (test_swap["carrier_swapped"] >= test_swap["order_approved_at"]) &
    (test_swap["delivered_swapped"] >= test_swap["carrier_swapped"])
)

test_swap["sequence_valid_after_swap"].value_counts()

,count
sequence_valid_after_swap,
True,23


**Analyst Commentary**

23 delivered orders showed `order_delivered_customer_date` preceding `order_delivered_carrier_date`, which is a logically impossible sequence. Evidence showed that swapping the values between these two columns resolves the sequence to a valid order (approval → carrier → delivery) for all 23 rows. This strongly suggests a column-swap bug in the data pipeline (nearly half of cases concentrated in July 2017) rather than 23 unrelated data entry errors.

Recommendation: Correct these 23 rows by swapping the two date fields, and flag the pipeline/ETL step for review around the July 2017 period specifically.

**Analyst Commentary**

Findings from this dataset should be considered reliable overall, with two known, diagnosed limitations: (1) 23 orders have evidence consistent with a correctable column-swap error affecting delivery timestamps; (2) `order_approved_at` shows identified chronological inconsistencies in a subset of records across ~15 months, affecting confidence in any approval-time-based or monthly-trend metrics until the field is remediated at the source.  Those remediations would require engineering support.

Since a data engineer isn't accessible in this project, aggregating seller-level metrics are likely only marginally affected given the small proportion of rows involved, but time-based/seasonal analyses should be interpreted cautiously given the clustering of these errors.



**Correcting the 23-Row Column-Swap Bug**

Before engineering delivery-related features, the confirmed column-swap error (see swap test above) must be corrected. This ensures delivery_time_days, carrier_time_hours, and related features are calculated from valid, chronologically consistent timestamps for all rows.

In [45]:
# Correct the 23 rows where order_delivered_carrier_date and order_delivered_customer_date were swapped
# (confirmed via the swap test above: all 23 rows resolve to a valid sequence once swapped)

swap_mask = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
print(f"Correcting {swap_mask.sum()} rows with swapped delivery timestamps.")

orders.loc[swap_mask, ["order_delivered_carrier_date", "order_delivered_customer_date"]] = \
    orders.loc[swap_mask, ["order_delivered_customer_date", "order_delivered_carrier_date"]].values

# Verify the fix: this should now return 0
(orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()


Correcting 23 rows with swapped delivery timestamps.


np.int64(0)

**Feature Engineering**

To support the business questions explored in subsequent notebooks, the following features were engineered from the cleaned orders data.

In [46]:
# delivery_time_days feature: measure overall delivery performance

orders["delivery_time_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

# approval_time_hours feature: measure approval delays

orders["approval_time_hours"] = (orders["order_approved_at"] - orders["order_purchase_timestamp"]).dt.total_seconds() / 3600

# carrier_time_hours feature: measure fulfillment delay before shipping

orders["carrier_time_hours"] = (orders["order_delivered_carrier_date"] - orders["order_approved_at"]).dt.total_seconds() / 3600

# days_late feature: quantify delivery lateness

orders["days_late"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

# is_late_delivery feature: flag late deliveries

orders["is_late_delivery"] = orders["days_late"].apply(
    lambda x: True if pd.notna(x) and x > 0
    else False if pd.notna(x) and x <= 0
    else pd.NA
)

In [47]:
# Creating small date dimensions

orders["purchase_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month
)

orders["purchase_month_name"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

orders["purchase_quarter"] = (
    orders["order_purchase_timestamp"]
    .dt.quarter
)

orders["purchase_weekday"] = (
    orders["order_purchase_timestamp"]
    .dt.day_name()
)

orders["order_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

**Validation**

After engineering new features, validation checks are performed to ensure that impossible or inconsistent timelines have not been introduced during cleaning.

In [48]:
# Table

orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,approval_time_hours,carrier_time_hours,days_late,is_late_delivery,purchase_year,purchase_month,purchase_month_name,purchase_quarter,purchase_weekday,order_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,0.178333,56.795833,-8.0,False,2017,10,October,4,Monday,2017-10-01
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,30.713889,11.109167,-6.0,False,2018,7,July,3,Tuesday,2018-07-01
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,0.276111,4.910278,-18.0,False,2018,8,August,3,Wednesday,2018-08-01
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,0.298056,89.900000,-13.0,False,2017,11,November,4,Saturday,2017-11-01
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,1.030556,21.434722,-10.0,False,2018,2,February,1,Tuesday,2018-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,8.0,0.000000,25.399444,-11.0,False,2017,3,March,1,Thursday,2017-03-01
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,22.0,0.194167,34.201389,-2.0,False,2018,2,February,1,Tuesday,2018-02-01
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,24.0,0.292500,29.802778,-6.0,False,2017,8,August,3,Sunday,2017-08-01
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,17.0,0.131667,89.978333,-21.0,False,2018,1,January,1,Monday,2018-01-01


In [49]:
# Datatypes

orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]
delivery_time_days,float64
approval_time_hours,float64


**Cleaning Summary**

*Customers:*

1. Converted ZIP code to string.
2. No duplicate customer IDs.
3. No missing values requiring action.

*Orders:*

1. Converted datetime columns.
2. Investigated missing timestamps.
3. Determined most missing timestamps were consistent with the order lifecycle.
4. Investigated chronological timestamp inconsistencies.
5. Corrected 23 rows by swapping two date fields.
6. Created eleven engineered features.
7. Validated chronological consistency.

*Engineered Features:*

- `delivery_time_days`
- `days_late`
- `approval_time_hours`
- `carrier_time_hours`
- `is_late_delivery`
- `purchase_year`
- `purchase_month`
- `purchase_month_name`
- `purchase_quarter`
- `purchase_weekday`
- `order_month`

*Outstanding Questions:*

1. Why do some delivered orders lack delivery timestamps?
2. What does "created" order status represent?
3. What business process creates "unavailable" orders?

**Analyst Commentary**

The cleaned dataset is now suitable for feature engineering and downstream analyses involving operational performance, customer satisfaction, seller performance, and product category priortization.

In [50]:
# Export cleaned CSVs

customers.to_csv("customers_clean.csv", index=False)
orders.to_csv("orders_clean.csv", index=False)